<a href="https://colab.research.google.com/github/Mario-Cattaneo/SemesterProject/blob/main/Splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import os
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.transforms import Resize, CenterCrop, ToTensor, Normalize, Compose
from PIL import Image
from collections import deque

# 1. Clone the repository if it doesn't exist
REPO_DIR = "imagenet-sample-images"
REPO_URL = "https://github.com/EliSchwartz/imagenet-sample-images.git"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# =====================================================================
# 2. Define the Edge (Communication Channel) Module
# =====================================================================
class CommunicationEdge(nn.Module):
    def __init__(self, source_id: str, target_id: str):
        super().__init__()
        self.source_id = source_id
        self.target_id = target_id
        self.encoder = nn.Identity()  # Extendable with trainable coders
        self.channel = nn.Identity()  # Extendable with differentiable noisy channels
        self.decoder = nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.channel(self.encoder(x)))

# =====================================================================
# 3. Define the Device Node Module
# =====================================================================
class DeviceNode(nn.Module):
    def __init__(self, node_id: str, weight: float):
        super().__init__()
        self.node_id = node_id
        self.weight = weight
        self.layers = nn.Sequential()
        self.edges = nn.ModuleDict()  # Outgoing edges
        self.predecessors = []        # List of predecessor node IDs
        self.downsample_factor = 1    # Estimated during compilation

    def add_successor(self, successor_node: 'DeviceNode'):
        edge_id = f"{self.node_id}->{successor_node.node_id}"
        self.edges[edge_id] = CommunicationEdge(self.node_id, successor_node.node_id)
        successor_node.predecessors.append(self.node_id)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

# =====================================================================
# 4. Define the General Distributed Graph Compiler & Executor
# =====================================================================
class DistributedGraph(nn.Module):
    def __init__(self):
        super().__init__()
        self.nodes = nn.ModuleDict()
        self.topological_levels = []

    def add_node(self, node_id: str, weight: float):
        self.nodes[node_id] = DeviceNode(node_id, weight)

    def add_edge(self, source_id: str, target_id: str):
        self.nodes[source_id].add_successor(self.nodes[target_id])

    def _get_in_channels(self, module: nn.Module) -> int:
        """Helper to dynamically find the expected input channels of a block."""
        for sub_module in module.modules():
            if isinstance(sub_module, nn.Conv2d):
                return sub_module.in_channels
        return 3  # Default fallback (RGB)

    def compile_and_allocate(self, model: nn.Module):
        """
        Validates the graph, sorts it topologically, and automatically
        allocates the model layers across the devices.
        """
        # 1. Topological Sort & Validation (Kahn's Algorithm)
        in_degree = {node_id: len(node.predecessors) for node_id, node in self.nodes.items()}
        queue = deque([node_id for node_id, deg in in_degree.items() if deg == 0])

        levels = []
        while queue:
            level_nodes = list(queue)
            levels.append(level_nodes)
            queue.clear()
            for node_id in level_nodes:
                for edge in self.nodes[node_id].edges.values():
                    target_id = edge.target_id
                    in_degree[target_id] -= 1
                    if in_degree[target_id] == 0:
                        queue.append(target_id)

        # Cycle Detection
        if sum(len(lvl) for lvl in levels) != len(self.nodes):
            raise ValueError("Graph contains cycles. Only Directed Acyclic Graphs (DAGs) are valid.")

        self.topological_levels = levels
        D = len(levels)  # Graph Depth

        # 2. Vertical Partitioning of VGG-16 Features
        features_layers = list(model.features.children())
        num_layers = len(features_layers)

        if D > num_layers:
            raise ValueError(f"Graph depth ({D}) exceeds available model layers ({num_layers}).")

        # Divide layers into D blocks
        block_size = num_layers // D
        for d, level_nodes in enumerate(levels):
            start_idx = d * block_size
            end_idx = (d + 1) * block_size if d < D - 1 else num_layers
            layer_block = nn.Sequential(*features_layers[start_idx:end_idx])

            for node_id in level_nodes:
                node = self.nodes[node_id]
                if d == D - 1:
                    # Terminal level gets the classifier as well
                    node.layers = nn.Sequential(
                        layer_block,
                        model.avgpool,
                        nn.Flatten(1),
                        model.classifier
                    )
                else:
                    node.layers = layer_block

        # 3. Estimate Downsampling Factors dynamically using correct channel sizes
        with torch.no_grad():
            for node in self.nodes.values():
                in_channels = self._get_in_channels(node.layers)
                dummy_in = torch.randn(1, in_channels, 224, 224)
                try:
                    test_layers = node.layers[0] if isinstance(node.layers[0], nn.Sequential) else node.layers
                    dummy_out = test_layers(dummy_in)
                    node.downsample_factor = dummy_in.shape[2] // dummy_out.shape[2]
                except Exception:
                    node.downsample_factor = 1

    def print_structure(self):
        print("\n" + "="*80)
        print("COMPILED DISTRIBUTED GRAPH TOPOLOGY")
        print("="*80)
        for d, level_nodes in enumerate(self.topological_levels):
            print(f"\n--- LEVEL {d} ---")
            for node_id in level_nodes:
                node = self.nodes[node_id]
                print(f"Device Node: '{node_id}' | Weight: {node.weight} | Downsample Factor: {node.downsample_factor}")
                if len(node.edges) > 0:
                    print(" -> Outgoing Edges:")
                    for edge_id in node.edges.keys():
                        print(f"    * {edge_id}")
        print("="*80 + "\n")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        node_outputs = {}

        # Traverse the graph level by level
        for d, level_nodes in enumerate(self.topological_levels):
            for node_id in level_nodes:
                node = self.nodes[node_id]

                # 1. Gather and Merge Inputs from Predecessors
                if d == 0:
                    node_input = x
                else:
                    incoming_tensors = []
                    for pred_id in node.predecessors:
                        edge_id = f"{pred_id}->{node_id}"
                        transmitted_tensor = self.nodes[pred_id].edges[edge_id](node_outputs[pred_id])
                        incoming_tensors.append(transmitted_tensor)

                    if len(incoming_tensors) == 1:
                        node_input = incoming_tensors[0]
                    else:
                        # Merge parallel branches by cropping out the downsampled overlap
                        scale = node.downsample_factor
                        pad_size = self.nodes[node.predecessors[0]].downsample_factor

                        # Calculate remaining overlap after downsampling
                        crop_rows = pad_size // scale

                        h_A = incoming_tensors[0].shape[2] - crop_rows

                        clean_A = incoming_tensors[0][:, :, 0:h_A, :]
                        clean_B = incoming_tensors[1][:, :, crop_rows:, :]
                        node_input = torch.cat([clean_A, clean_B], dim=2)

                # 2. Execute Node Layers
                node_outputs[node_id] = node(node_input)

                # 3. Branching / Splitting for Successors
                if len(node.edges) > 1:
                    successors = [self.nodes[edge.target_id] for edge in node.edges.values()]
                    total_weight = sum(succ.weight for succ in successors)

                    B, C, H, W = node_outputs[node_id].shape
                    h_A = int(round((successors[0].weight / total_weight) * H))

                    # Enforce that the overlap size matches the downsampling factor of the successors
                    pad_size = successors[0].downsample_factor

                    # Slice with dynamic overlap
                    slice_A = node_outputs[node_id][:, :, 0 : h_A + pad_size, :]
                    slice_B = node_outputs[node_id][:, :, h_A - pad_size : H, :]

                    node_outputs[node_id] = slice_A  # Route to successor 1
                    node_outputs[f"{node_id}_split2"] = slice_B

        terminal_node_id = self.topological_levels[-1][0]
        return node_outputs[terminal_node_id]

# =====================================================================
# 5. Self-Contained ImageNet Loader
# =====================================================================
def load_imagenet_samples(k=20):
    all_files = [f for f in os.listdir(REPO_DIR) if f.endswith(".JPEG")]
    unique_wnids = sorted(list(set(f.split("_")[0] for f in all_files)))
    wnid_to_idx = {wnid: idx for idx, wnid in enumerate(unique_wnids)}

    selected_files = random.sample(all_files, min(k, len(all_files)))

    transform = Compose([
        Resize(224),
        CenterCrop(224),
        ToTensor(),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    images, targets, names = [], [], []
    for filename in selected_files:
        parts = filename.replace(".JPEG", "").split("_")
        wnid = parts[0]
        class_name = " ".join(parts[1:])
        class_idx = wnid_to_idx[wnid]
        img_path = os.path.join(REPO_DIR, filename)
        try:
            img = Image.open(img_path).convert('RGB')
            images.append(transform(img))
            targets.append(class_idx)
            names.append(class_name)
        except Exception as e:
            print(f"Failed to load {filename}: {e}")

    return torch.stack(images), torch.tensor(targets), names

# =====================================================================
# 6. Main Evaluation Pipeline
# =====================================================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running evaluation on: {device}\n")

    # Load Pretrained VGG16
    print("Loading pretrained VGG16 model...")
    vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT).to(device).eval()

    # Define DAG Topology
    dist_graph = DistributedGraph()
    dist_graph.add_node('P', weight=1.0)
    dist_graph.add_node('A', weight=0.6)
    dist_graph.add_node('B', weight=0.4)
    dist_graph.add_node('S', weight=1.0)

    dist_graph.add_edge('P', 'A')
    dist_graph.add_edge('P', 'B')
    dist_graph.add_edge('A', 'S')
    dist_graph.add_edge('B', 'S')

    # Compile and allocate layers automatically
    dist_graph.compile_and_allocate(vgg16)
    dist_graph.to(device)
    dist_graph.print_structure()

    # Load k random ImageNet samples
    k_samples = 50
    print(f"Loading {k_samples} random ImageNet samples from local repository...")
    images, targets, names = load_imagenet_samples(k=k_samples)
    images, targets = images.to(device), targets.to(device)

    # Run Inference
    print("\nRunning inference...")
    with torch.no_grad():
        out_base = vgg16(images)
        out_dag = dist_graph(images)

    # Calculate Accuracies
    correct_base = 0
    correct_dag = 0

    for i in range(len(names)):
        target_label = targets[i].item()
        pred_base = out_base[i].argmax().item()
        pred_dag = out_dag[i].argmax().item()

        if pred_base == target_label: correct_base += 1
        if pred_dag == target_label: correct_dag += 1

    # Print final accuracies
    total = len(names)
    print("\n" + "="*50)
    print(f"EVALUATION RESULTS ON {total} IMAGENET SAMPLES")
    print("="*50)
    print(f"Baseline Accuracy (Unsplit): {correct_base/total*100:.2f}%")
    print(f"Distributed Graph Accuracy:  {correct_dag/total*100:.2f}%")
    print("="*50)

if __name__ == "__main__":
    main()

Running evaluation on: cpu

Loading pretrained VGG16 model...

COMPILED DISTRIBUTED GRAPH TOPOLOGY

--- LEVEL 0 ---
Device Node: 'P' | Weight: 1.0 | Downsample Factor: 4
 -> Outgoing Edges:
    * P->A
    * P->B

--- LEVEL 1 ---
Device Node: 'A' | Weight: 0.6 | Downsample Factor: 2
 -> Outgoing Edges:
    * A->S
Device Node: 'B' | Weight: 0.4 | Downsample Factor: 2
 -> Outgoing Edges:
    * B->S

--- LEVEL 2 ---
Device Node: 'S' | Weight: 1.0 | Downsample Factor: 4

Loading 50 random ImageNet samples from local repository...

Running inference...

EVALUATION RESULTS ON 50 IMAGENET SAMPLES
Baseline Accuracy (Unsplit): 74.00%
Distributed Graph Accuracy:  70.00%
